# Hair App Canonical Crop Test

이 노트북은 **1단계 얼굴 crop만** 검증한다. Pixel3DMM, PIPNet, FaRL segmentation, normal/UV, FLAME fitting은 실행하지 않는다.

처리 내용: 사진별 얼굴 검출 → 얼굴 위치·크기 통일 → roll 제거 → 512×512 crop → 양방향 affine metadata 저장 → 원본/crop 시각 비교. yaw와 pitch는 보존한다.

## 1. crop detector 설치

In [ ]:
!pip -q install 'git+https://github.com/FacePerceiver/facer.git@ddd35c76ff840174b8a5403ad1c1255e37b8782b'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. Google Drive 및 경로 설정

입력은 `MyDrive/hair_app/inputs/`, 결과는 별도 `MyDrive/hair_app/crop_test_512/`에 저장한다. 원본은 수정하지 않는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
INPUT_DIR = Path('/content/drive/MyDrive/hair_app/inputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/hair_app/crop_test_512')
OUTPUT_SIZE = 512
BBOX_MARGIN = 1.42
DETECTION_THRESHOLD = 0.5

input_files = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg', '.png'}
)
print('inputs:', len(input_files), [path.name for path in input_files])
assert input_files, f'입력 이미지 없음: {INPUT_DIR}'

## 3. 사진별 canonical crop engine

In [ ]:
import json
import math
import shutil
from dataclasses import dataclass

import facer
import numpy as np
from PIL import Image, ImageOps

@dataclass(frozen=True)
class FaceObservation:
    bbox_xyxy: tuple
    left_eye_xy: tuple
    right_eye_xy: tuple
    score: float

def load_oriented_rgb(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert('RGB')

def transform_points(matrix, points):
    points = np.asarray(points, dtype=np.float64)
    homogeneous = np.concatenate([points, np.ones((len(points), 1))], axis=1)
    return (matrix @ homogeneous.T).T[:, :2]

class RetinaFaceDetector:
    def __init__(self, device, threshold):
        self.device = device
        self.detector = facer.face_detector('retinaface/mobilenet', device=device)
        self.detector.threshold = threshold

    def detect(self, image):
        pixels = np.asarray(image, dtype=np.uint8).copy()
        tensor = facer.hwc2bchw(torch.from_numpy(pixels)).to(self.device)
        with torch.inference_mode():
            faces = self.detector(tensor)
        if faces['scores'].numel() == 0:
            raise RuntimeError('RetinaFace returned no face')
        index = int(torch.argmax(faces['scores']).item())
        bbox = faces['rects'][index].detach().cpu().tolist()
        points = faces['points'][index].detach().cpu().tolist()
        score = float(faces['scores'][index].detach().cpu().item())
        eye_a = tuple(float(value) for value in points[0])
        eye_b = tuple(float(value) for value in points[1])
        left_eye, right_eye = sorted((eye_a, eye_b), key=lambda point: point[0])
        return FaceObservation(
            bbox_xyxy=tuple(float(value) for value in bbox),
            left_eye_xy=left_eye,
            right_eye_xy=right_eye,
            score=score,
        )

def canonical_crop(image, observation, output_size=512, bbox_margin=1.42):
    x1, y1, x2, y2 = observation.bbox_xyxy
    assert x2 > x1 and y2 > y1, observation.bbox_xyxy
    bbox_width, bbox_height = x2 - x1, y2 - y1
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    crop_side = max(bbox_width, bbox_height) * bbox_margin
    scale = output_size / crop_side
    eye_dx = observation.right_eye_xy[0] - observation.left_eye_xy[0]
    eye_dy = observation.right_eye_xy[1] - observation.left_eye_xy[1]
    roll_degrees = math.degrees(math.atan2(eye_dy, eye_dx))
    angle = math.radians(roll_degrees)
    cosine, sine = math.cos(angle), math.sin(angle)
    output_center = output_size / 2

    source_to_crop = np.array([
        [scale*cosine, scale*sine, output_center-scale*cosine*center_x-scale*sine*center_y],
        [-scale*sine, scale*cosine, output_center+scale*sine*center_x-scale*cosine*center_y],
        [0, 0, 1],
    ], dtype=np.float64)
    crop_to_source = np.linalg.inv(source_to_crop)
    affine = tuple(float(value) for value in crop_to_source[:2].reshape(-1))
    crop = image.transform(
        (output_size, output_size),
        Image.Transform.AFFINE,
        affine,
        resample=Image.Resampling.BICUBIC,
        fillcolor=(0, 0, 0),
    )
    eyes_in_crop = transform_points(
        source_to_crop, [observation.left_eye_xy, observation.right_eye_xy]
    )
    warnings = []
    if observation.score < 0.55:
        warnings.append('low_detection_confidence')
    if min(bbox_width, bbox_height) < 64:
        warnings.append('low_source_face_resolution')
    if abs(roll_degrees) > 25:
        warnings.append('extreme_roll_recapture_preferred')
    metadata = {
        'version': '0.1',
        'source_size': list(image.size),
        'output_size': [output_size, output_size],
        'bbox_margin': bbox_margin,
        'face_occupancy_target': 1 / bbox_margin,
        'roll_degrees_removed': roll_degrees,
        'source_to_crop': source_to_crop.tolist(),
        'crop_to_source': crop_to_source.tolist(),
        'bbox_xyxy': list(observation.bbox_xyxy),
        'left_eye_xy': list(observation.left_eye_xy),
        'right_eye_xy': list(observation.right_eye_xy),
        'eyes_in_crop': eyes_in_crop.tolist(),
        'detection_score': observation.score,
        'warnings': warnings,
    }
    return crop, metadata

print('canonical crop engine: READY')

## 4. crop 8장 생성

기존 `crop_test_512` 결과만 지우고 다시 만든다. `inputs` 원본은 건드리지 않는다.

In [ ]:
assert OUTPUT_ROOT.name == 'crop_test_512', f'안전하지 않은 출력 경로: {OUTPUT_ROOT}'
shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
crop_dir = OUTPUT_ROOT / 'cropped'
meta_dir = OUTPUT_ROOT / 'crop_meta'
crop_dir.mkdir(parents=True)
meta_dir.mkdir(parents=True)

detector = RetinaFaceDetector(DEVICE, DETECTION_THRESHOLD)
manifest_items = []

for index, source_path in enumerate(input_files):
    image = load_oriented_rgb(source_path)
    try:
        observation = detector.detect(image)
    except Exception as error:
        raise RuntimeError(f'{source_path.name} 얼굴 검출 실패: {error}') from error
    crop, metadata = canonical_crop(
        image, observation, output_size=OUTPUT_SIZE, bbox_margin=BBOX_MARGIN
    )
    derived_name = f'{index:05d}.jpg'
    crop.save(crop_dir / derived_name, quality=95)
    item = {'source_name': source_path.name, 'derived_name': derived_name, **metadata}
    (meta_dir / f'{index:05d}.json').write_text(
        json.dumps(item, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    manifest_items.append(item)
    print(
        f'CROP PASS {source_path.name} -> {derived_name} '
        f'score={observation.score:.3f} roll={metadata["roll_degrees_removed"]:.2f} '
        f'warnings={metadata["warnings"]}'
    )

manifest = {
    'version': '0.1',
    'engine': 'hair_app_per_image_canonical_crop',
    'count': len(manifest_items),
    'output_size': OUTPUT_SIZE,
    'bbox_margin': BBOX_MARGIN,
    'items': manifest_items,
}
(meta_dir / 'manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('CANONICAL CROP COMPLETE:', len(manifest_items))

## 5. 원본과 crop을 한 쌍씩 비교

왼쪽 원본에는 검출 bbox와 눈 선을 표시한다. 오른쪽 crop에서 헤어라인·눈·코·입·턱·필요한 귀가 보이는지 직접 확인한다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(len(manifest_items), 2, figsize=(13, 4 * len(manifest_items)), squeeze=False)
for index, item in enumerate(manifest_items):
    source = load_oriented_rgb(INPUT_DIR / item['source_name'])
    crop = Image.open(crop_dir / item['derived_name']).convert('RGB')
    x1, y1, x2, y2 = item['bbox_xyxy']
    left_eye, right_eye = item['left_eye_xy'], item['right_eye_xy']

    axes[index, 0].imshow(source)
    axes[index, 0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red', linewidth=2))
    axes[index, 0].plot(
        [left_eye[0], right_eye[0]], [left_eye[1], right_eye[1]],
        color='cyan', marker='o', linewidth=2,
    )
    axes[index, 0].set_title(f'원본: {item["source_name"]}\nscore={item["detection_score"]:.3f}')
    axes[index, 1].imshow(crop)
    axes[index, 1].set_title(
        f'crop: {item["derived_name"]} | roll 제거={item["roll_degrees_removed"]:.1f}°\n'
        f'warnings={item["warnings"]}'
    )
    axes[index, 0].axis('off'); axes[index, 1].axis('off')
plt.tight_layout(); plt.show()

## 6. 계산 검증

자동 검사는 개수·크기·눈 수평·affine 역변환만 확인한다. 얼굴 부위가 충분히 포함됐는지는 위 그림을 사람이 확인해야 한다.

In [ ]:
crop_files = sorted(crop_dir.glob('*.jpg'))
meta_files = sorted(path for path in meta_dir.glob('*.json') if path.name != 'manifest.json')
assert len(crop_files) == len(input_files) == len(meta_files)

for crop_path, item in zip(crop_files, manifest_items):
    assert Image.open(crop_path).size == (512, 512)
    forward = np.asarray(item['source_to_crop'])
    inverse = np.asarray(item['crop_to_source'])
    np.testing.assert_allclose(inverse @ forward, np.eye(3), atol=1e-7)
    eyes = np.asarray(item['eyes_in_crop'])
    assert abs(float(eyes[0, 1] - eyes[1, 1])) < 1e-5

print('AUTOMATIC CROP CHECKS: PASS')
print('다음 단계로 가지 말고, 위 원본/crop 쌍의 얼굴 coverage를 먼저 확인하세요.')
print('saved:', OUTPUT_ROOT)